# Environment


In [1]:
import dotenv

# Store your API keys in the .env file
dotenv.load_dotenv()

True

# Data Generation Setup


In [2]:
import json
import os
import re
import random
import logging
from abc import ABC, abstractmethod
import time
from typing import List, Dict, Optional, Any, Set, cast

from google import genai
import anthropic
import openai
from tenacity import (
    retry,
    wait_exponential,
    stop_after_attempt,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(asctime)s [%(filename)s:%(lineno)d] %(message)s",
)

if "GOOGLE_API_KEY" not in os.environ:
    logging.warning(
        "GOOGLE_API_KEY environment variable not set. GeminiProvider will fail."
    )
if "ANTHROPIC_API_KEY" not in os.environ:
    logging.warning(
        "ANTHROPIC_API_KEY environment variable not set. AnthropicProvider will fail."
    )
if "OPENAI_API_KEY" not in os.environ:
    logging.warning(
        "OPENAI_API_KEY environment variable not set. OpenAIProvider will fail."
    )


class BaseLLMProvider(ABC):
    """Abstract base class for LLM API providers."""

    @abstractmethod
    def generate(self, prompt: str) -> str:
        """
        Generates text given a prompt, with built-in retries.

        Args:
            prompt: The input prompt string.

        Returns:
            The LLM's text response.
        """
        pass


class GeminiProvider(BaseLLMProvider):
    """LLM Provider for Google Gemini."""

    def __init__(self, model_name: str = "gemini-1.5-pro-latest"):
        if "GOOGLE_API_KEY" not in os.environ:
            raise EnvironmentError("GOOGLE_API_KEY environment variable not set.")
        self.client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
        self.model = model_name
        logging.info(f"Initialized GeminiProvider with model: {model_name}")

    @retry(
        wait=wait_exponential(multiplier=10, max=180),
        stop=stop_after_attempt(8),
    )
    def generate(self, prompt: str) -> str:
        response = self.client.models.generate_content(
            model=self.model, contents=prompt
        )
        if not response.parts:
            logging.warning(f"Gemini response blocked or empty: {response}.")
            raise ValueError("Gemini response was empty or blocked.")
        return response.text  # type: ignore


class AnthropicProvider(BaseLLMProvider):
    """LLM Provider for Anthropic Claude."""

    def __init__(self, model_name: str = "claude-3-5-sonnet-20240620"):
        if "ANTHROPIC_API_KEY" not in os.environ:
            raise EnvironmentError("ANTHROPIC_API_KEY environment variable not set.")
        self.client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        self.model_name = model_name
        logging.info(f"Initialized AnthropicProvider with model: {model_name}")

    @retry(
        wait=wait_exponential(multiplier=10, max=180),
        stop=stop_after_attempt(8),
    )
    def generate(self, prompt: str) -> str:
        message = self.client.messages.create(
            model=self.model_name,
            max_tokens=1500,
            messages=[{"role": "user", "content": prompt}],
        )
        if not message.content:
            logging.warning(f"Anthropic response empty: {message}.")
            raise ValueError("Anthropic response was empty.")
        return message.content[0].text  # type: ignore


class OpenAIProvider(BaseLLMProvider):
    """LLM Provider for OpenAI GPT."""

    def __init__(self, model_name: str = "gpt-4o"):
        if "OPENAI_API_KEY" not in os.environ:
            raise EnvironmentError("OPENAI_API_KEY environment variable not set.")
        self.client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model_name = model_name
        logging.info(f"Initialized OpenAIProvider with model: {model_name}")

    @retry(
        wait=wait_exponential(multiplier=10, max=180),
        stop=stop_after_attempt(8),
    )
    def generate(self, prompt: str) -> str:
        completion = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
        )
        response = completion.choices[0].message.content
        if not response:
            logging.warning("OpenAI response empty.")
            raise ValueError("OpenAI response was empty.")
        return response


def _get_base_generator_prompt(
    is_support: bool, statement: str, num_arguments: int = 4
) -> str:
    """
    Generates a prompt for an LLM to create a ranked list of arguments
    with clearly distinct quality levels.

    Args:
        is_support: Whether the arguments should support (True) or attack (False) the statement.
        statement: The statement to generate arguments about.
        num_arguments: The number of arguments to generate (must be 3, 4, or 5).

    Returns:
        str: A formatted prompt string.

    Raises:
        ValueError: If num_arguments is not between 3 and 5 (inclusive).
    """
    if not 3 <= num_arguments <= 5:
        raise ValueError(
            f"Number of arguments must be between 3 and 5, but got {num_arguments}"
        )

    stance = "support" if is_support else "attack"

    if is_support:
        argument_type = "positive evidence, benefits, or logical reasons in favor"
    else:
        argument_type = (
            "refutations, counter-evidence, critiques, or logical reasons against"
        )

    quality_structures = {
        3: [
            "**(Strongest):** A very valid, factually accurate, and logically sound argument that directly and compellingly addresses the statement.",
            "**(Medium):** A partially justified argument that has a clear flaw or is significantly less relevant than the first, but is not completely nonsensical.",
            "**(Flawed/Nonsensical):** A completely invalid argument. It should be nonsensical or internally contradictory, but still framed as an argument with the correct stance.",
        ],
        4: [
            "**(Strongest):** A very valid, factually accurate, and logically sound argument that directly and compellingly addresses the statement.",
            "**(Medium):** A plausible argument that is noticeably weaker. It might be a valid point but less relevant, a broad generalization, or rely on a weak or unstated assumption.",
            "**(Weak):** A clearly flawed argument. It might contain a common logical fallacy (e.g., straw man, ad hominem, appeal to emotion), a demonstrable factual inaccuracy, or be only tangentially related to the statement's core.",
            "**(Flawed/Nonsensical):** A completely invalid argument. It should be nonsensical or internally contradictory, but still framed as an argument with the correct stance.",
        ],
        5: [
            "**(Strongest):** A very valid, factually accurate, and logically sound argument that directly and compellingly addresses the statement.",
            "**(High-Medium):** A plausible, common argument that is clearly weaker than the first. It might be a valid point but less relevant, or a minor generalization.",
            "**(Low-Medium):** A noticeably weaker argument. It might be logically valid but rest on a very niche or contestable premise, or be a significant oversimplification.",
            "**(Weak):** A clearly flawed argument. It might contain a common logical fallacy (e.g., straw man, appeal to emotion), a demonstrable factual inaccuracy, or be only tangentially related.",
            "**(Flawed/Nonsensical):** A completely invalid argument. It should be nonsensical or internally contradictory, but still framed as an argument with the correct stance.",
        ],
    }

    structure_list = quality_structures[num_arguments]
    structure_text = "\n".join(
        f"{i + 1}. {desc}" for i, desc in enumerate(structure_list)
    )

    prompt = f"""You are an expert in logic, rhetoric, and argumentation. Your task is to generate a ranked set of arguments.
Your goal is to create {num_arguments} short, one-sentence arguments that {stance} the following statement:
"{statement}"

Produce a numbered list of exactly {num_arguments} arguments, strictly ranked from most convincing to least convincing. The quality drop between each argument must be distinct and unambiguous. Even if the arguments were shuffled, a person should be able to easily re-order them from strongest to weakest.

Follow this quality structure for your {num_arguments} arguments:
{structure_text}

When generating the arguments, focus on providing {argument_type}.
Crucially, all {num_arguments} arguments, including the weakest, must clearly maintain the '{stance}' stance and be on-topic.

Output *only* the numbered list of {num_arguments} arguments. Do not include any other text, commentary, or labels (like "Strongest" or "Medium")."""

    return prompt


def get_generator_prompt(
    is_support: bool,
    statement: str,
    num_arguments: int,
    feedback_history: Optional[List[str]] = None,
) -> str:
    """
    Gets the base generator prompt and appends feedback from previous
    failed attempts, if any.
    """
    base_prompt = _get_base_generator_prompt(is_support, statement, num_arguments)

    if not feedback_history:
        return base_prompt

    feedback_text = "\n\n---\n".join(feedback_history)
    retry_prompt = f"""
---
PREVIOUS ATTEMPT(S) FAILED VERIFICATION.
A separate AI verifier was given your arguments in a random order and failed to re-rank them correctly. This means the quality distinction between your arguments was not clear or 'distinct and unambiguous' enough.

Details from failed verification(s):
{feedback_text}

Please try again. Generate a *new* set of {num_arguments} arguments.
Focus on making the quality differences *extremely* clear, as defined in the original prompt. The verifier must be able to easily sort them.
Output *only* the new numbered list of {num_arguments} arguments.
"""
    return base_prompt + retry_prompt


def get_verification_prompt(
    is_support: bool, statement: str, shuffled_map: Dict[str, str]
) -> str:
    """
    Generates a prompt for the Verifier LLM to rank shuffled arguments.
    """
    stance = "support" if is_support else "attack"
    num_args = len(shuffled_map)
    arg_list_text = "\n".join(
        f"{label}: {text}" for label, text in shuffled_map.items()
    )

    return f"""You are a precise and objective expert in logic and argument quality.
Your task is to rank the following {num_args} arguments related to the statement:
"{statement}"

The arguments are all intended to {stance} the statement, but they are of different quality levels (e.g., some are strong, some are flawed, some are nonsensical). They are presented below in a random order, identified by labels.

Arguments:
{arg_list_text}

Please rank these arguments from STRONGEST (most valid, relevant, and sound) to WEAKEST (most flawed, irrelevant, or nonsensical).

Output *only* a comma-separated list of the labels in your ranked order (from strongest to weakest).
For example, if you have labels A, B, C, D and you think C is strongest, then A, then D, then B is weakest, you must output:
C, A, D, B
"""


def parse_generator_output(raw_text: str, num_arguments: int) -> Optional[List[str]]:
    """
    Parses the numbered list output from the Generator LLM.
    """
    try:
        # Find all lines starting with "1. ", "2. ", etc.
        arguments = re.findall(r"^\s*\d+\.\s*(.+)", raw_text, re.MULTILINE)

        # Strip leading/trailing whitespace from each found argument
        arguments = [arg.strip() for arg in arguments]

        if len(arguments) == num_arguments:
            return arguments
        else:
            logging.warning(
                f"Parser: Expected {num_arguments} args, found {len(arguments)}. Output: '{raw_text}'"
            )
            return None
    except Exception as e:
        logging.error(f"Parser error in parse_generator_output: {e}")
        return None


def parse_verifier_output(
    raw_text: str, expected_labels: Set[str]
) -> Optional[List[str]]:
    """
    Parses the comma-separated label list from the Verifier LLM.
    """
    try:
        # Clean up the output, remove potential markdown, etc.
        clean_text = re.sub(r"`", "", raw_text).strip()

        # Handle cases where the LLM might have added extra text
        # Find the first line that looks like a list of labels
        match = re.search(r"^([A-Z](?:,\s*[A-Z])+)$", clean_text, re.MULTILINE)
        if match:
            clean_text = match.group(1)

        labels = [label.strip() for label in clean_text.split(",")]

        if set(labels) == expected_labels and len(labels) == len(expected_labels):
            return labels
        else:
            logging.warning(
                f"Verifier Parser: Mismatch. Got {set(labels)}, expected {expected_labels}. Output: '{raw_text}'"
            )
            return None
    except Exception as e:
        logging.error(f"Parser error in parse_verifier_output: {e}")
        return None


def load_processed_statements(filepath: str) -> Set[str]:
    """
    Reads a JSONL file and returns a set of 'statement' fields
    to avoid re-processing them.
    """
    processed_statements = set()
    if not os.path.exists(filepath):
        return processed_statements

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        data = json.loads(line)
                        if "statement" in data:
                            processed_statements.add(data["statement"])
                    except json.JSONDecodeError:
                        logging.warning(f"Skipping malformed JSON line: {line}")
    except IOError as e:
        logging.error(f"Could not read checkpoint file {filepath}: {e}")

    return processed_statements


def append_to_jsonl(filepath: str, data: Dict[str, Any]):
    """
    Appends a new data record to a JSONL file.
    """
    try:
        with open(filepath, "a", encoding="utf-8") as f:
            json.dump(data, f)
            f.write("\n")
    except IOError as e:
        logging.error(f"Failed to write to output file {filepath}: {e}")


def _build_failed_verification_feedback(
    shuffle_num: int,
    statement: str,
    stance_str: str,
    shuffled_map: Dict[str, str],
    ground_truth_ranking: List[str],
    correct_label_order: List[str],
    verifier_ranking: Optional[List[str]],
    raw_verifier_output: str,
) -> str:
    """
    Helper function to build the readable feedback string.
    """

    # 1. Show what the verifier saw
    shuffled_list_str = "\n".join(
        f"  - {label}: {text}" for label, text in shuffled_map.items()
    )

    # 2. Show the correct ranking
    correct_ranking_str = "\n".join(
        f"  {i + 1}. {label} (GT Rank {i + 1}): {ground_truth_ranking[i]}"
        for i, label in enumerate(correct_label_order)
    )

    # 3. Show the verifier's incorrect ranking
    if verifier_ranking:
        verifier_ranking_str = "\n".join(
            f"  {i + 1}. {label} (Verifier Rank {i + 1}): {shuffled_map[label]}"
            for i, label in enumerate(verifier_ranking)
        )
    else:
        verifier_ranking_str = (
            f"Verifier output was unparseable: '{raw_verifier_output}'"
        )

    return f"""
Verification (Shuffle {shuffle_num + 1}) FAILED:
  - Statement: "{statement}"
  - Stance: {stance_str}

Arguments sent to Verifier (shuffled):
{shuffled_list_str}

Correct Ranking (Ground Truth):
{correct_ranking_str}

Verifier's (Incorrect) Ranking:
{verifier_ranking_str}
"""


def generate_and_verify_arguments(
    statement: str,
    is_support: bool | None,
    generator_llm: BaseLLMProvider,
    verifier_llm: BaseLLMProvider,
    num_arguments: int = 4,
    max_attempts: int = 3,
    num_shuffles: int = 2,
) -> Optional[Dict[str, Any]]:
    """
    Main pipeline function to generate, verify, and retry argument generation.
    """
    feedback_history: List[str] = []

    is_effective_support = is_support
    if is_support is None:
        random.seed(hash(statement) % (2**32))
        is_effective_support = random.choice([True, False])
    is_effective_support = cast(bool, is_effective_support)

    stance_str = "support" if is_effective_support else "attack"

    for attempt in range(max_attempts):
        logging.info(
            f"--- Attempt {attempt + 1}/{max_attempts} for statement: '{statement[:50]}...' ---"
        )

        # 1. GENERATE ARGUMENTS
        try:
            gen_prompt = get_generator_prompt(
                is_effective_support, statement, num_arguments, feedback_history
            )
            raw_gen_output = generator_llm.generate(gen_prompt)
            generated_args = parse_generator_output(raw_gen_output, num_arguments)

            if not generated_args:
                logging.warning("Generation parsing failed.")
                feedback_history.append(
                    f"Attempt {attempt + 1} failed: Generator output was unparseable or had the wrong number of arguments. Output: '{raw_gen_output}'"
                )
                continue  # Try again

            # This is our ground-truth ranking
            ground_truth_ranking = generated_args
            logging.info(
                f"Generated {len(ground_truth_ranking)} arguments (ground truth)."
            )

        except Exception as e:
            logging.error(f"Generation API call failed: {e}")
            feedback_history.append(
                f"Attempt {attempt + 1} failed: Generator API call raised an exception: {e}"
            )
            continue  # Try again

        # 2. VERIFICATION STAGE
        verification_passed = True
        failed_verification_details = ""

        for shuffle_num in range(num_shuffles):
            logging.info(
                f"Running verification shuffle {shuffle_num + 1}/{num_shuffles}..."
            )

            # --- Create shuffled map and correct order ---
            # Original order: [0, 1, 2, 3] (indices of ground_truth_ranking)
            original_indices = list(range(num_arguments))

            # Shuffled order: e.g., [2, 0, 3, 1]
            random.seed(shuffle_num)
            shuffled_indices = random.sample(original_indices, num_arguments)

            # Labels: ['A', 'B', 'C', 'D']
            labels = [chr(ord("A") + i) for i in range(num_arguments)]

            # Map labels to shuffled arguments for the prompt
            # e.g., {'A': arg_at_index_2, 'B': arg_at_index_0, ...}
            shuffled_map = {
                label: ground_truth_ranking[original_idx]
                for label, original_idx in zip(labels, shuffled_indices)
            }

            # Map original indices to their new random label
            # e.g., {0: 'B', 1: 'D', 2: 'A', 3: 'C'}
            original_index_to_label_map = {
                original_idx: label
                for label, original_idx in zip(labels, shuffled_indices)
            }

            # The correct re-ordered list of labels
            # e.g., ['B', 'D', 'A', 'C']
            correct_label_order = [
                original_index_to_label_map[i] for i in original_indices
            ]
            # --- End shuffle logic ---

            raw_ver_output = ""
            verifier_ranking = None
            try:
                ver_prompt = get_verification_prompt(
                    is_effective_support, statement, shuffled_map
                )
                raw_ver_output = verifier_llm.generate(ver_prompt)
                verifier_ranking = parse_verifier_output(raw_ver_output, set(labels))

                if verifier_ranking == correct_label_order:
                    logging.info(f"Shuffle {shuffle_num + 1} PASSED.")
                    continue  # This shuffle passed, try the next

                else:
                    # Verification failed for this shuffle
                    verification_passed = False
                    failed_verification_details = _build_failed_verification_feedback(
                        shuffle_num,
                        statement,
                        stance_str,
                        shuffled_map,
                        ground_truth_ranking,
                        correct_label_order,
                        verifier_ranking,
                        raw_ver_output,
                    )
                    logging.warning(failed_verification_details)
                    break  # Stop shuffles and move to retry

            except Exception as e:
                verification_passed = False
                failed_verification_details = (
                    f"Verification (Shuffle {shuffle_num + 1}) API call failed: {e}"
                )
                logging.error(failed_verification_details)
                break  # Stop shuffles and move to retry

        # 3. CHECK RESULT & RETURN OR ADD FEEDBACK
        if verification_passed:
            logging.info(f"*** SUCCESS for statement: '{statement[:50]}...' ***")
            return {
                "statement": statement,
                "is_support": is_support,
                "is_effective_support": is_effective_support,
                "num_arguments": num_arguments,
                "ground_truth_ranking": ground_truth_ranking,
            }
        else:
            # Add the failure details for the next generator prompt
            feedback_history.append(failed_verification_details)

    # If loop finishes, all attempts failed
    logging.error(
        f"--- FAILED all {max_attempts} attempts for statement: '{statement[:50]}...' ---"
    )
    return None


def process_dataset(
    dataset: List[Dict[str, Any]],
    generator_llm: BaseLLMProvider,
    verifier_llm: BaseLLMProvider,
    num_arguments: int,
    output_filepath: str,
    limit: int | None = None,
    sample_delay: float | None = None,
) -> List[Dict[str, Any]]:
    """
    Iterates over a dataset and runs the generation/verification pipeline.
    """
    # Load checkpoint
    processed_statements = load_processed_statements(output_filepath)
    if processed_statements:
        logging.info(
            f"Loaded {len(processed_statements)} processed statements from {output_filepath}. They will be skipped."
        )

    # Start after the last successful sample
    data_to_process = dataset
    for i, item in enumerate(dataset):
        if item["statement"] in processed_statements:
            logging.info(
                f"Skipping already processed statement at index {i}: '{item['statement'][:50]}...'"
            )
            data_to_process = dataset[i + 1 :]

    successful_samples = []
    for i, item in enumerate(data_to_process):
        total_successful = len(successful_samples) + len(processed_statements)
        logging.info(
            f"Processing sample #{i + 1}, with {total_successful} successful samples so far."
        )
        statement = item["statement"]
        is_support = item["is_correct"]

        result = generate_and_verify_arguments(
            statement=statement,
            is_support=is_support,
            generator_llm=generator_llm,
            verifier_llm=verifier_llm,
            num_arguments=num_arguments,
        )

        if result:
            successful_samples.append(result)
            append_to_jsonl(output_filepath, result)

        if sample_delay:
            logging.info(f"Sleeping for {sample_delay} seconds before next sample...")

            time.sleep(sample_delay)

        total_successful = len(successful_samples) + len(processed_statements)
        if limit and total_successful >= limit:
            logging.info(f"Reached processing limit of {limit} successful samples.")
            break

    return successful_samples

# Base Data Loading


In [3]:
import csv

# Load argllms_plus_plus_staging/training_true_false_uncertain.csv
with open("../Data/Training/training_true_false_uncertain.csv", "r") as f:
    reader = csv.DictReader(f)
    training_data = [row for row in reader]
claims = [row["claim"] for row in training_data]

# Load training_true_false_uncertain_with_arguments.json
with open("../Data/Training/training_true_false_uncertain_with_arguments.json", "r") as f:
    arguments_data = json.load(f)
argument_claims = [item["claim"] for item in arguments_data]

unique_claims = set(claims) - set(argument_claims)

dataset = []
for sample in training_data:
    if sample["claim"] in unique_claims:
        dataset.append(
            {
                "statement": sample["claim"],
                "is_correct": None
                if sample["valid"] == "Uncertain"
                else sample["valid"] == "True",
            }
        )
random.seed(42)
random.shuffle(dataset)
print(f"Dataset size for processing: {len(dataset)}")

Dataset size for processing: 502


In [4]:
dataset

[{'statement': 'Russia consumes more tea on average than Americans do.',
  'is_correct': True},
 {'statement': 'The Gulf federal state called the United States of America is a member of the United Nations, has valuable natural resources, contains a global city and international hub.',
  'is_correct': False},
 {'statement': 'The existence of extraterrestrial life capable of interstellar travel has been deliberately concealed from the public by a coordinated effort among world governments.',
  'is_correct': None},
 {'statement': 'Plucking a gray hair will result in two more growing in its place.',
  'is_correct': False},
 {'statement': 'Toronto gets more snowfall than Montreal.',
  'is_correct': False},
 {'statement': 'Sherlock Holmes lives on Baker Street in London in real life.',
  'is_correct': False},
 {'statement': 'The Dutch are, on average, shorter than Americans.',
  'is_correct': False},
 {'statement': 'The Viking civilization had contact with the Native Americans before Christo

# Data Generation

Adjust models and parameters as needed.


In [ ]:
OUTPUT_FILE = "../Data/Training/Ranking/gen_gpt-5_ver_sonnet-4-5_arguments.jsonl"

# --- LLM Choice ---
try:
    generator_model = OpenAIProvider(model_name="gpt-5-2025-08-07")
    verifier_model = AnthropicProvider(model_name="claude-sonnet-4-5-20250929")
    NUMBER_OF_ARGUMENTS_TO_GENERATE = 4

    logging.info(f"Starting dataset processing with {len(dataset)} items...")
    logging.info(f"Verified results will be saved to: {OUTPUT_FILE}")

    final_verified_dataset = process_dataset(
        dataset=dataset,
        generator_llm=generator_model,
        verifier_llm=verifier_model,
        num_arguments=NUMBER_OF_ARGUMENTS_TO_GENERATE,
        output_filepath=OUTPUT_FILE,
        limit=800,
        sample_delay=5.0,
    )

    # --- PRINT RESULTS ---
    print("\n" + "=" * 80)
    print(
        f"COMPLETED: Added {len(final_verified_dataset)} new verified samples to {OUTPUT_FILE}."
    )
    print("=" * 80 + "\n")

    if final_verified_dataset:
        print("--- Samples added in this run ---")
        for i, sample in enumerate(final_verified_dataset):
            print(f"\n--- New Sample {i + 1} ---")
            print(f"Statement: {sample['statement']}")
            print(f"Stance: {'Support' if sample['is_support'] else 'Attack'}")
            print("Verified Argument Ranking (Strongest to Weakest):")
            for j, arg in enumerate(sample["ground_truth_ranking"]):
                print(f"  {j + 1}. {arg}")
    else:
        print(
            "No new samples were added in this run (they may have already been processed)."
        )

except (EnvironmentError, ImportError) as e:
    logging.error(
        f"Failed to initialize LLM providers. Please check your API keys and dependencies: {e}"
    )
except Exception as e:
    logging.error(f"An unexpected error occurred: {e}")

INFO 2026-04-30 16:06:38,314 [3851128910.py:114] Initialized OpenAIProvider with model: gpt-5-2025-08-07


INFO 2026-04-30 16:06:38,329 [3851128910.py:88] Initialized AnthropicProvider with model: claude-sonnet-4-5-20250929
INFO 2026-04-30 16:06:38,329 [3742493685.py:11] Starting dataset processing with 150 items...
INFO 2026-04-30 16:06:38,330 [3742493685.py:12] Verified results will be saved to: ../Data/Evaluation/Ranking/true_false_uncertain_gen_gpt-5_ver_sonnet-4-5.jsonl
INFO 2026-04-30 16:06:38,331 [3851128910.py:579] Loaded 149 processed statements from ../Data/Evaluation/Ranking/true_false_uncertain_gen_gpt-5_ver_sonnet-4-5.jsonl. They will be skipped.
INFO 2026-04-30 16:06:38,331 [3851128910.py:587] Skipping already processed statement at index 0: 'Number of Imams Reza Shah believed in exceed numbe...'
INFO 2026-04-30 16:06:38,331 [3851128910.py:587] Skipping already processed statement at index 1: 'A Trek 9000 requires an anchor in order to park....'
INFO 2026-04-30 16:06:38,331 [3851128910.py:587] Skipping already processed statement at index 2: 'A cauliflower farmer would prosper


COMPLETED: Added 0 new verified samples to ../Data/Evaluation/Ranking/true_false_uncertain_gen_gpt-5_ver_sonnet-4-5.jsonl.

No new samples were added in this run (they may have already been processed).
